# Latent Escape Demonstration

A single component is followed from conventional screening through lot-relative evidence, temporal drift, 168 h forecast, safety decision, explanation, and lead-time analysis.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from src.pipeline import screen_dataframe, PipelineArtifacts
from src.evaluation import escape_matrix

ROOT=Path.cwd()
data=ROOT/'data/processed/module_A_dataset.csv'
artifacts=PipelineArtifacts(ROOT/'models/anomaly/model.joblib', ROOT/'models/forecast/model.joblib', ROOT/'models/calibration/ood_profile.joblib')
full=pd.read_csv(data)
test=full[full['split'].astype(str).eq('test')].copy()


In [ ]:
# Find an absolute-pass component that is eventually defective.
labels=(test.groupby('part_id').agg(
    absolute_fail=('absolute_fail_168h','max'),
    future_defective=('latent_defect_label','max'),
    lot_id=('lot_id','first'),
    parameter=('parameter','first'),
    component_family=('component_family','first')
).reset_index())
escapes=labels[(labels.absolute_fail==0)&(labels.future_defective==1)]
assert len(escapes), 'No latent-escape case is present in this test split.'
part_id=str(escapes.iloc[0].part_id)
case=test[test.part_id.eq(part_id)].copy()
case

## 1. Raw measurements

These are the observations available for the selected component. The 168 h value is shown here only because this notebook is an **evaluation demonstration**; the operational model is run at 24 h without using the future value as an input.

In [ ]:
display(case[['part_id','lot_id','component_family','parameter','unit','value_0h','value_24h','value_96h','value_168h','absolute_limit_upper']].head())

## 2. Conventional screening

If the 168 h value is still below the absolute upper limit, conventional limit screening reports **PASS**. This is the escape condition we want the intelligent layer to expose earlier.

In [ ]:
row=case.iloc[0]
limit=float(row.absolute_limit_upper) if pd.notna(row.absolute_limit_upper) else np.nan
actual=float(row.value_168h)
print('Absolute-limit decision:', 'PASS' if np.isfinite(limit) and actual <= limit else 'FAIL')
print('168 h value:', actual, ' limit:', limit)

## 3. Run the complete system using only the operational 24 h origin

The full test split is passed through the pipeline so the lot statistics are meaningful. The 168 h ground truth remains held out from model input.

In [ ]:
run=screen_dataframe(test, ROOT/'reports/demo_latent_escape', artifacts=artifacts, as_of_h=24, auto_train_missing=False)
result=run.screening[run.screening.part_id.astype(str).eq(part_id)]
display(result[['part_id','decision','risk_score','confidence','ood_status','failure_risk','anomaly_risk','uncertainty_score']])

## 4. Lot context and temporal evidence

In [ ]:
a=run.anomaly[run.anomaly.part_id.astype(str).eq(part_id)].iloc[0]
print('Lot-relative evidence:', a.get('population_evidence'))
print('Parameter-isolation evidence:', a.get('parameter_isolation_evidence'))
print('Cross-parameter evidence:', a.get('multivariate_component_score'))
print('Temporal evidence:', a.get('temporal_evidence'))

## 5. Module-B forecast

In [ ]:
f=run.forecast[run.forecast.part_id.astype(str).eq(part_id)].head(1)
display(f[['part_id','parameter','selected_forecast_model','prediction_168h','prediction_lower','prediction_upper','failure_risk','predicted_limit_exceedance']])

## 6. Final explanation

The explanation deliberately separates measured facts, model findings, safety policy, counterfactual reasoning, and recommended next test. It contains no requirement for the reader to understand Isolation Forest, SHAP, or other ML terminology.

In [ ]:
e=run.explanations[run.explanations.part_id.astype(str).eq(part_id)].iloc[0]
for col in ['summary','specific_counterfactual','lead_time','recommended_next_test','facts','model_findings','policy_reasoning','pattern_attribution_json']:
    print(f'\n--- {col} ---')
    print(e[col])

## 7. Lead time

The demonstration treats 24 h as the first operational screening opportunity. A defective part flagged at that point has a potential 144 h lead over the 168 h endpoint. This is a secondary engineering metric, not a replacement for the SIH metrics.

In [ ]:
print('Lead time estimate:', e['lead_time'])
print('Final decision:', e['decision'])

## 8. Evidence reminder

This notebook demonstrates the evaluation workflow on synthetic/reference data. It must not be presented as proof of exact ISRO hardware behaviour or guaranteed spacecraft-failure reduction.